In [5]:
!pip install gymnasium numpy

import numpy as np
import gymnasium as gym
from gymnasium import spaces


class AdaptivePlannerEnv(gym.Env):
    def __init__(self):
        super().__init__()

        # Actions:
        # 0 = do task 1
        # 1 = do task 2
        # 2 = skip
        # 3 = take break
        self.action_space = spaces.Discrete(4)

        # State:
        # [time, energy, task1_deadline, task2_deadline, tasks_completed]
        self.observation_space = spaces.Box(
            low=0, high=100, shape=(5,), dtype=np.float32
        )

    def reset(self, seed=None, options=None):
        self.time = 0
        self.energy = 100

        # Initial tasks
        self.tasks = [
            {"deadline": 5, "done": 0, "priority": 2},
            {"deadline": 8, "done": 0, "priority": 1},
        ]

        return self._get_state(), {}

    def _get_state(self):
        return np.array([
            self.time,
            self.energy,
            self.tasks[0]["deadline"],
            self.tasks[1]["deadline"],
            sum(t["done"] for t in self.tasks)
        ], dtype=np.float32)

    def step(self, action):
        reward = 0

        # --- ACTION LOGIC ---
        if action == 0 and self.tasks[0]["done"] == 0:
            self.tasks[0]["done"] = 1
            reward += 10 + self.tasks[0]["priority"] * 2

        elif action == 1 and self.tasks[1]["done"] == 0:
            self.tasks[1]["done"] = 1
            reward += 8 + self.tasks[1]["priority"] * 2

        elif action == 2:
            reward -= 2  # idle penalty

        elif action == 3:
            self.energy = min(100, self.energy + 15)
            reward += 3  # break reward

        # --- TIME & ENERGY UPDATE ---
        self.time += 1
        self.energy -= 5

        # --- DEADLINE PENALTY ---
        for t in self.tasks:
            if t["done"] == 0 and self.time > t["deadline"]:
                reward -= 5

        # --- ENERGY PENALTY (NEW ADDITION) ---
        if self.energy < 20:
            reward -= 3

        # --- RANDOM TASK ARRIVAL ---
        if np.random.rand() < 0.3:
            self.tasks.append({
                "deadline": self.time + 5,
                "done": 0,
                "priority": np.random.randint(1, 3)
            })

        # Episode ends after fixed time
        done = self.time >= 12

        return self._get_state(), reward, done, False, {}


# ------------------ TEST RUN ------------------

env = AdaptivePlannerEnv()
state, _ = env.reset()

print("Starting Environment Test...\n")

for i in range(10):
    action = env.action_space.sample()
    state, reward, done, _, _ = env.step(action)
    print(f"Step {i} → State: {state}, Reward: {reward}")

    if done:
        print("\nEpisode Finished")
        break

Starting Environment Test...

Step 0 → State: [ 1. 95.  5.  8.  1.], Reward: 14
Step 1 → State: [ 2. 90.  5.  8.  1.], Reward: -2
Step 2 → State: [ 3. 85.  5.  8.  1.], Reward: 0
Step 3 → State: [ 4. 95.  5.  8.  1.], Reward: 3
Step 4 → State: [ 5. 90.  5.  8.  1.], Reward: -2
Step 5 → State: [ 6. 85.  5.  8.  1.], Reward: 0
Step 6 → State: [ 7. 80.  5.  8.  2.], Reward: 10
Step 7 → State: [ 8. 75.  5.  8.  2.], Reward: -2
Step 8 → State: [ 9. 70.  5.  8.  2.], Reward: -7
Step 9 → State: [10. 80.  5.  8.  2.], Reward: -2


Adaptive Planner RL Environment

Problem:
Managing tasks, deadlines, and energy is challenging in real life.

Environment Design:

State → time, energy, deadlines, completion
Actions → perform task, skip, take break

Reward Logic:

+10 → task completed
-5 → missed deadline
-2 → idle
+3 → break
-3 → low energy penalty

Features:

Random task arrival
Energy-based decision making
Real-world simulation

Adaptive Planner RL Environment

Problem:
Managing tasks, deadlines, and energy efficiently is a real-world challenge.

Environment Design:

State: time, energy, task deadlines, completion status
Actions: select task, skip, take break

Reward Function:

+10 → task completed
-5 → missed deadline
-2 → idle
+3 → break

Key Features:

Dynamic task arrival (random tasks)
Energy-based decision making
Simulates real-world interruptions